# Train the crop disease model (Colab GPU, works in the browser or the VS Code Colab extension)

**VS Code:** Select Kernel > Colab > New Colab Server > GPU (T4). The code runs on Colab's machine, not your PC,
so the first cells clone the repo there, and the model is sent back by uploading it to Hugging Face Hub.

**Keys:** in the browser, add them in the 🔑 Secrets panel (`ROBOFLOW_API_KEY`, `HF_TOKEN`, and `GITHUB_TOKEN` if the
repo is private). In VS Code, or if a secret is missing, the cell asks you to paste it instead. Keys are never saved.

Output: `model_<version>/` with `model.tflite`, `classes.json`, `calibration.json`, `preprocess.json`, `metrics.json`
(see `contracts/model_io.md`).

In [ ]:
REPO = "your-github-user/cropdisease_detector"
BRANCH = "model-pipeline"
VERSION = "v0.1"
FINE_TUNE_EPOCHS = 8                    # 0 = heads only (fast); 8 is a good start on a T4
HF_MODEL_REPO = "cropdiseasedetectors/crop-disease-model"
!nvidia-smi -L

In [ ]:
import os
from getpass import getpass

def secret(name, required=True):
    """Colab Secrets panel when available (browser), otherwise ask (VS Code)."""
    try:
        from google.colab import userdata
        value = userdata.get(name)
        if value:
            return value
    except Exception:
        pass
    value = getpass(f"{name} (leave empty to skip): " if not required else f"{name}: ")
    return value or None

token = secret("GITHUB_TOKEN", required=False)  # only for a private repo
url = f"https://{token}@github.com/{REPO}.git" if token else f"https://github.com/{REPO}.git"
!rm -rf /content/repo && git clone -q --depth 1 -b {BRANCH} {url} /content/repo
%cd /content/repo
!pip install -q -r ml/requirements.txt huggingface_hub

## 1. PlantVillage (lab photos: our 13 classes, plus other crops for other_plant)

In [ ]:
!git clone -q --depth 1 --filter=blob:none --sparse https://github.com/spMohanty/PlantVillage-Dataset /content/pv
!cd /content/pv && git sparse-checkout set --no-cone "/raw/color/*"
!python ml/src/prepare_plantvillage.py /content/pv/raw/color /content/data

## 2. Roboflow (field photos)

Fill `ml/configs/roboflow_datasets.json` first (in the repo, then push). Run the class listing, check the mapping, then download.

In [ ]:
os.environ["ROBOFLOW_API_KEY"] = secret("ROBOFLOW_API_KEY")
!python ml/src/download_roboflow.py --list-classes --cache /content/rf

In [ ]:
!python ml/src/download_roboflow.py --out /content/data --cache /content/rf
!for d in /content/data/*/; do echo "$(ls $d | wc -l) $(ls $d | grep -c '^rf_') $(basename $d)"; done  # total, field, label

## 3. Train, calibrate, export

In [ ]:
!python ml/src/train.py --data /content/data --out /content/model_{VERSION} --version {VERSION} --fine-tune-epochs {FINE_TUNE_EPOCHS}

In [ ]:
import json
m = json.load(open(f"/content/model_{VERSION}/metrics.json"))
for head, s in m["per_head"].items():
    print(head, "macro-F1", s["macro_f1"], "| field photos:", s.get("field_photos_only", "none"),
          "| below 0.70:", s["share_below_0.70_confidence"])

In [ ]:
# Send the model back: upload to Hugging Face Hub (private). Then download it from the repo's "Files" tab,
# or on your PC: huggingface-cli download YOUR-HF-USER/crop-disease-model --revision v0.1 --local-dir backend/models/v0.1
from huggingface_hub import HfApi
api = HfApi(token=secret("HF_TOKEN"))
api.create_repo(HF_MODEL_REPO, private=True, exist_ok=True)
api.upload_folder(folder_path=f"/content/model_{VERSION}", repo_id=HF_MODEL_REPO,
                  ignore_patterns=["splits.json"], commit_message=f"model {VERSION}")
api.create_tag(HF_MODEL_REPO, tag=VERSION, exist_ok=True)
print(f"https://huggingface.co/{HF_MODEL_REPO}/tree/{VERSION}")

In [ ]:
# Browser only (optional): download a zip directly instead.
!cd /content && zip -qr model_{VERSION}.zip model_{VERSION}
from google.colab import files
files.download(f"/content/model_{VERSION}.zip")